# CIFAR-10 Autoencoder with GPU Acceleration - Final Project Report
**Group: 15**

---

# **1. Problem Description**

## 1.1 Problem Statement and Motivation

Trong thời đại xã hội phát triển như vũ bão, AI dần khẳng định vị thế chiến lược trong đời sống và việc làm của vô số người. Nhằm ngày càng mở rộng và cải tiến công nghệ, học máy là một trong các kỹ thuật phổ biến nhất được áp dụng trong việc nghiên cứu và ứng dụng AI. Bên cạnh các đột phá, còn tồn tại nhiều hạn chế và thách thức trong việc áp dụng, triển khai phương pháp này. 

Một thách thức đặc trưng của học máy là làm thế nào để tự động phát hiện các biểu diễn dữ liệu tốt, phản ánh cấu trúc tiềm ẩn của dữ liệu mà không phụ thuộc vào nhãn dán có sẵn. Vì suy cho cùng, các nhãn vẫn mang yếu tố chủ quan trong chúng, nên chắc chắn không thể khái quát hóa lên cao, mà chính sự tổng quát hóa mới giúp ta khám phá và thúc đẩy các giới hạn trong lĩnh vực nghiên cứu AI nói riêng, khoa học máy tính nói chung. 

Do đó, nhóm em triển khai một hệ thống để học các đặc trưng một cách không giám sát dựa trên autoencoder, sử dụng dữ liệu mẫu là bài toán phân loại ảnh với bộ dữ liệu CIFAR-10, giúp đương đầu khó khăn phải tìm ra đặc trưng, các mẫu (pattern) của dữ liệu mà có tính tổng quát, khái quát.

Phương pháp học có giám sát truyền thống huấn luyện mô hình bằng các ví dụ có nhãn. Song, autoencoder cố gắng giải quyết vấn đề trên bằng cách cố gắng tìm ra các đặc trưng nổi bật nhất của tập dữ liệu mà không phụ thuộc vào nhãn. Bằng cách này, ta có thể phát hiện các đặc trưng bền vững và tổng quát hơn so với cách tiếp cận thông thường.

**Hệ thống có hai giai đoạn chính:**

**Giai đoạn 1 – Học đặc trưng không giám sát:**

- Huấn luyện một autoencoder tích chập để trích xuất các đặc trưng ý nghĩa nhất từ ảnh.
- Bộ mã hóa (encoder) nén ảnh 32×32×3 thành vector đặc trưng 8.192 chiều. Không sử dụng nhãn trong giai đoạn huấn luyện này. Mạng học được các mẫu thị giác quan trọng (cạnh, kết cấu, hình dạng, v.v.).

**Giai đoạn 2 – Phân loại có giám sát:**

- Trích xuất đặc trưng từ encoder đã huấn luyện cho toàn bộ tập ảnh.
- Huấn luyện bộ phân loại SVM trên các đặc trưng đã học với nhãn lớp.
- Đánh giá hiệu quả phân loại trên tập kiểm thử.

## 1.2 CIFAR-10 Dataset Overview

![image.png](attachment:image.png)

**Đặc điểm bộ dữ liệu:**
- **Kích thước ảnh:** 32×32 pixel (màu RGB)
- **Số lớp:** 10 loại (máy bay, ô tô, chim, mèo, nai, chó, ếch, ngựa, tàu thủy, xe tải)
- **Tập huấn luyện:** 50.000 ảnh (5.000 ảnh mỗi lớp)
- **Tập kiểm thử:** 10.000 ảnh (1.000 ảnh mỗi lớp)
- **Định dạng:** Nhóm sẽ làm với tệp nhị phân với giá trị pixel uint8 [0-255]
- **Cách tổ chức của các tập tin nhị phân**: Sẽ có 6 file nhị phân, lần lượt là `data_batch_1.bin` đến `data_batch_5.bin` và `test_batch.bin`, mỗi file có cấu trúc như sau:
    ```
    <1 x label><3072 x pixel>
    ...
    <1 x label><3072 x pixel>
    ```

**Các bước tiền xử lý:**
- Tải các tệp CIFAR-10 nhị phân theo yêu cầu, hoặc 5 batch huấn luyện cùng lúc, hoặc 1 batch kiểm thử
- Phân tích định dạng nhị phân: 1 byte nhãn + 3.072 byte (3x32×32) mỗi ảnh
- Giúp mô hình học nhanh và ổn định hơn, tránh hiện tượng gradient quá lớn hoặc quá nhỏ, đồng thời đảm bảo các giá trị đầu vào nằm trong cùng một khoảng, phù hợp với hàm kích hoạt ReLU, ta chuẩn hóa dữ liệu uint8 [0-255] sang float [0-1].
- Vì mỗi ảnh đã được lưu dưới dạng 1024 bytes đầu tiên là red channel, 1024 bytes tiếp theo là green channel và cuối cùng là blue. Ở mỗi channel dữ liệu được lưu theo row-major order. Dạng dữ liệu này đã phù với việc tính toán ma trận trong huấn luyện huẩn luyện mô hình (`tensor [NCHW]`) nên ta tiến hành load trực tiếp dữ liệu không cần chuyển đổi thứ tự.
- Đồng thời các hàm cơ bản của một dataloader cũng được triển khai như `shuffle` (xáo trộn vị trí các mẫu trong data), `load_batch` (chia data thành các batch để tối ưu training).

## 1.3 Autoencoder Architecture

**Tổng quan kiến trúc của Autoencoder:**

![image-2.png](attachment:image-2.png)

```
Input: (32×32×3) → Encoder → Latent: (8×8×128) = 8,192 đặc trưng → Decoder → Output: (32×32×3)
```

Autoencoder có 2 phần chính là **Encoder** và **Decoder**. Encoder được xem là phần để `mã hóa` dữ liệu (ở đây là ảnh) thành 1 dạng biểu diễn khác và dùng Decoder để `tái tạo` lại ảnh ban đầu từ dữ liệu đã được mã hóa. 

    Dưới đây là cấu trúc chi tiết về dimension ở các layers của mô hình:

**Encoder (Downsampling):**
- Input: (32, 32, 3)
- Conv2D(256 filters, 3×3, padding=1) + ReLU → (32, 32, 256)
- MaxPool2D(2×2, stride=2) → (16, 16, 256)
- Conv2D(128 filters, 3×3, padding=1) + ReLU → (16, 16, 128)
- MaxPool2D(2×2, stride=2) → (8, 8, 128)
- **Latent representation: (8, 8, 128) = 8,192 dimensions**

**Decoder (Upsampling):**
- Latent: (8, 8, 128)
- Conv2D(128 filters, 3×3, padding=1) + ReLU → (8, 8, 128)
- Upsample2D(2×2) → (16, 16, 128)
- Conv2D(256 filters, 3×3, padding=1) + ReLU → (16, 16, 256)
- Upsample2D(2×2) → (32, 32, 256)
- Conv2D(3 filters, 3×3, padding=1) → (32, 32, 3)
- Output: (32, 32, 3)

    - Cấu trúc của encoder: 
        - Các convolution layers, các filter được sử dụng để trích xuất, nhận biết các đặc trưng đặc biệt, quan trọng của bức ảnh.
        - Các maxpool layers dùng để giảm chiều của dữ liệu (ví dụ `[32, 32, 256]` → `[16, 16, 128]`). Thông qua quá trình này, số lượng phép tính mà mô hình phải thực hiện được giảm đi, đồng thời nó cũng bắt mô hình chỉ giữ lại các đặc trưng nổi bật, quan trọng nhất.
        > Thông qua quá trình này, chiều của dữ liệu được tăng lên, nhưng đồng thời các đặc trưng của ảnh được lưu ở lại nhiều hơn, điều rất quan trọng cho việc tái tạo lại ảnh.
    
    - Cấu trúc của latent space:
        - Ảnh input ban đầu có  $32 \times 32 \times 3 = 3,072$ giá trị nhưng khi biểu diễn ở latent space lại có tới $8 \times 8 \times 128 = 8,192$ giá trị.
        - Ở bước này dữ liệu không còn có thể xem như ảnh ban đầu. Thay vì biểu diễn ở các giá trị pixel rời rạc thì ở chiều dữ liệu cao hơn này, mỗi mẫu thể hiện được nhiều thông tin hơn ví dụ: "*có xuất hiện cánh máy bay*", "*có bánh xe tải*",...\
    
    - Cấu trúc của decoder: 
        - Khối decoder được dùng để tái tạo lại ảnh ban đầu, ta có thể nhận thấy cấu trúc của khối này như đối xứng so với encoder. Ngược lại với maxpool ta dùng Upsample, thay đổi số filter của convolution để tái tạo lại chiều cấu trúc ảnh ban đầu.


## 1.4 Mục tiêu

**Hiệu năng:**
- Thời gian huấn luyện autoencoder cho 50.000 ảnh huấn luyện: < 20 phút
- Thời gian trích xuất đặc trưng (50.000 ảnh huấn luyện + 10.000 ảnh test): < 30 giây
- Độ chính xác phân loại kiểm thử: 60-65%

**Kiến thức và trải nghiệm:**
- Hiện thực các phép toán deep learning từ đầu bằng C++/CUDA
- Thành thạo hơn về lập trình GPU: kernel, quản lý bộ nhớ, tối ưu hóa
- Hiểu và cài đặt được kiến trúc autoencoder ở mức độ cơ bản
- Biết quan sát và tối ưu hóa bằng cách áp dụng các phương pháp song song hóa lên GPU
- Biết tư duy, tìm tòi để cải tiến các phiên bản chạy trên GPU một cách có hệ thống

**Tiêu chí thành công:**
- Pipeline hoàn chỉnh, có thể xử lí từ ảnh gốc đến ra kết quả phân loại cho phiên bản cài bằng CPU, GPU
- Các sự cài đặt của module, layers, ... đều vượt qua unit test kiểm thử, đảm bảo tính đúng đắn cơ bản
- Cài đặt đúng, có kiểm chứng đối chiếu với phiên bản CPU sơ khai
- Đo lường được mức độ tăng tốc thông qua tối ưu GPU
- Độ chính xác phân loại đạt tối thiểu 60-65%
- Có ghi chú tài liệu rõ ràng về các chiến lược tối ưu và kết quả tương ứng


# **2. Implementation phases**

## **2.1 CPU baseline implementation**

Động lực tăng tốc bằng GPU:

Huấn luyện trên CPU quá chậm với 50.000 ảnh.
Song song hóa trên GPU giúp rút ngắn thời gian huấn luyện từ hàng giờ xuống vài phút.
Tối ưu hóa hệ thống đạt tốc độ nhanh hơn CPU ít nhất 20 lần.
Đây là yếu tố then chốt để triển khai thực tế và thử nghiệm hiệu quả.

### **2.1.1 Objectives**

**What we aimed to achieve in this phase:**
- Establish a working CPU baseline implementation of the autoencoder
- Validate the complete data pipeline for CIFAR-10 dataset
- Implement all neural network layers from scratch (Conv2D, ReLU, MaxPool, Upsample)
- Create training loop with forward/backward passes
- Measure baseline performance metrics (time, loss, memory)
- Verify correctness of implementation

**Why this phase is necessary:**
- Provides reference implementation to verify correctness of GPU versions
- Establishes baseline performance metrics for speedup comparisons
- Helps understand computational bottlenecks before GPU optimization
- Validates that the architecture can learn meaningful features
- Ensures end-to-end pipeline works before parallelization

_Placeholder for CPU implementation results:_
- Training time per epoch and total training time
- Final reconstruction loss
- Sample reconstructed images (original vs reconstructed)
- Memory usage statistics

### **2.1.2 Implementation Details**

**Data Pipeline:**
- Created CIFAR10Loader class to handle binary file loading
- Implemented batch generation with configurable size
- Added data shuffling for each epoch
- Normalized pixel values from uint8 [0-255] to float [0-1]
- Organized data as NCHW tensors for efficient processing

**Layer Implementations:**

1. **Conv2D Layer:**
   - 3×3 convolution with padding and stride support
   - Xavier weight initialization for training stability
   - Nested loops over batch, output channels, input channels, spatial dimensions
   - Boundary handling for padding

2. **ReLU Activation:**
   - Element-wise activation: f(x) = max(0, x)
   - Simple forward pass with no learnable parameters
   - Gradient: df/dx = 1 if x > 0, else 0

3. **MaxPool Layer:**
   - 2×2 pooling windows with stride 2
   - Downsamples spatial dimensions by factor of 2
   - Tracks max indices for backpropagation

4. **Upsample Layer:**
   - Nearest neighbor interpolation
   - Doubles spatial dimensions (2× upsampling)
   - No learnable parameters

5. **MSE Loss:**
   - Mean Squared Error between output and target
   - Loss = (1/N) * Σ(output - target)²
   - Used for reconstruction objective

**Training Loop Structure:**
```
Hyperparameters:
- Batch size: 32
- Epochs: 20
- Learning rate: 0.001
- Optimizer: SGD

For each epoch:
    Shuffle training data
    For each batch:
        Forward pass: input → encoder → decoder → output
        Compute loss: MSE(output, input)
        Backward pass: compute gradients
        Update weights: weight -= learning_rate * gradient
    Save weights and metrics
```

**Key Code Snippets:**

_Placeholder for 2-3 critical code snippets showing:_
- Convolution forward pass implementation
- Training loop structure
- Weight update mechanism

### **2.1.3 Results**

_Placeholder for detailed results from CPU training_

### **2.1.4 Key Takeaways**

_Placeholder for lessons learned:_
- What did you learn about the algorithm?
- What insights guided your GPU implementation?
- What were the main computational bottlenecks?
- Which operations took the most time?


## **2.2 GPU Naive Implementation**

### **2.2.1 Objectives**

**What we aimed to achieve in this phase:**
- Port CPU code to GPU with basic parallelization
- Verify correctness of GPU kernels against CPU baseline
- Establish baseline GPU performance metrics
- Implement naive GPU kernels without advanced optimizations
- Validate device memory management

**Why this phase is necessary:**
- Establishes working GPU implementation before optimization
- Provides performance baseline for measuring optimization impact
- Validates correctness of parallel algorithms
- Identifies initial bottlenecks for optimization

### **2.2.2 Implementation Details**


**Parallelization Strategy:**

_Placeholder for parallelization approach:_
- How operations were mapped to GPU threads
- Thread block dimensions chosen
- Grid configuration strategy

**Kernel Designs:**

1. **Convolution Kernel:**
   - Each thread computes one output pixel
   - Thread loops over kernel and input channels
   - Uses global memory for reads/writes
   - Handles padding boundaries

2. **ReLU Kernel:**
   - Each thread processes one element
   - Simple element-wise operation

3. **MaxPooling Kernel:**
   - Each thread computes one output element
   - Finds maximum in 2×2 window

4. **Upsampling Kernel:**
   - Each thread computes one output pixel
   - Maps coordinates to input (nearest neighbor)

**Memory Management:**
- Device memory allocation for weights, activations, gradients
- Host-to-device and device-to-host transfers
- Proper memory cleanup and error checking

**Key Code Snippets:**

_Placeholder for kernel signatures and launch configurations_



### **2.2.3 Results**

_Placeholder for GPU naive results:_
- Training time per epoch and total time
- Speedup over CPU baseline (table and chart)
- GPU memory usage
- Verification that outputs match CPU (error metrics)

### **2.2.4 Profiling Analysis**

_Placeholder for profiling results:_
- Time spent in each kernel type
- Memory bandwidth utilization
- Initial bottleneck identification

### **2.2.5 Key Takeaways**

_Placeholder for lessons learned:_
- What was surprisingly fast or slow?
- Where do you see optimization opportunities?
- Which kernels are memory-bound vs compute-bound?



## **2.3 GPU Optimized Implementation - Version 1**

### **2.3.1 Optimization Focus**

_Specify optimization category: e.g., Memory Optimization, Shared Memory, etc._

### **2.3.2 Objectives**

**What we aimed to achieve:**
- _Placeholder: What specific optimization(s) you targeted_
- _Placeholder: Expected performance improvement_
- _Placeholder: Which bottleneck you addressed_

### **2.3.3 Implementation Details**

**Optimization Technique(s) Applied:**

_Placeholder for detailed explanation:_
- What optimization was implemented (e.g., shared memory tiling)
- Why this optimization should help
- Implementation approach and challenges
- Changes to kernel design

**Key Code Snippets:**

_Placeholder for optimized kernel code_

### **2.3.4 Results**

_Placeholder for optimization v1 results:_
- Training time comparison with naive GPU version
- Speedup over previous phase (incremental)
- Cumulative speedup over CPU baseline
- Performance metrics (bandwidth, occupancy)
- Profiling comparison: before vs after

### **2.3.5 Analysis**

_Placeholder for analysis:_
- Why did this optimization work (or not work as expected)?
- What did profiling reveal?
- What's the next bottleneck to address?

### **2.3.6 Key Takeaways**

_Placeholder for lessons learned:_
- Main insights from this optimization
- Applicability to other problems
- Trade-offs encountered


## **2.4 GPU Optimized Implementation - Version 2** _(if applicable)_

### **2.4.1 Optimization Focus**

_Specify optimization category: e.g., Kernel Fusion, Advanced Techniques, etc._

### **2.4.2 Objectives**

_Placeholder for second optimization objectives_

### **2.4.3 Implementation Details**

_Placeholder for second optimization implementation details_

### **2.4.4 Results**

_Placeholder for second optimization results_

### **2.4.5 Analysis**

_Placeholder for second optimization analysis_

### **2.4.6 Key Takeaways**

_Placeholder for second optimization lessons learned_

---

_Note: Add additional optimization version sections as needed following the same structure_


## **2.5 SVM Integration**


### **2.5.1 Objectives**

**What we aimed to achieve:**
- Extract features from trained encoder for all images
- Train SVM classifier on learned features with labels
- Evaluate end-to-end classification performance
- Validate quality of learned features through classification accuracy

### **2.5.2 Implementation Details**

**Feature Extraction:**
- Loaded trained encoder weights from autoencoder training
- Ran encoder forward pass (without decoder) on all images
- Extracted latent representations: (N, 8, 8, 128) → (N, 8192)
- Processed 50,000 training images → 50,000 × 8,192 features
- Processed 10,000 test images → 10,000 × 8,192 features

**LIBSVM Integration:**
- Used LIBSVM library for SVM training and prediction
- Converted feature vectors to LIBSVM node format
- Interface between C++ feature extraction and LIBSVM C API

**Hyperparameter Selection:**
- **Kernel:** RBF (Radial Basis Function)
- **C parameter:** 10.0 (regularization)
- **Gamma:** auto (1 / num_features = 1 / 8192 ≈ 0.000122)
- **Cache size:** 100 MB

**Key Code Snippets:**

_Placeholder for:_
- Feature extraction code
- SVM training interface
- Prediction and evaluation code

### **2.5.3 Results**

_Placeholder for SVM results:_
- Feature extraction time (50K train + 10K test)
- SVM training time
- Overall test classification accuracy
- Per-class accuracy breakdown (table)
- Confusion matrix (visualization)
- Training set accuracy for reference

**Expected Results Format:**

| Metric | Value |
|--------|-------|
| Training samples | 50,000 |
| Test samples | 10,000 |
| Feature dimension | 8,192 |
| Feature extraction time | _TBD_ |
| SVM training time | _TBD_ |
| Training accuracy | _TBD_ |
| Test accuracy | _TBD_ |

### **2.5.4 Analysis**

_Placeholder for analysis:_
- Which classes are easiest to classify?
- Which classes are hardest to classify?
- What does the confusion matrix reveal about misclassifications?
- How does accuracy compare to expectations (60-65%)?
- What patterns of confusion exist (e.g., cat vs dog)?

### **2.5.5 Key Takeaways**

_Placeholder for lessons learned:_
- Quality of learned features from unsupervised training
- Effectiveness of two-stage approach (autoencoder + SVM)
- Advantages and limitations of this pipeline
- Comparison with end-to-end supervised approaches


# **3. Comprehensive Performance Analysis**

## 3.1 Performance Comparison Table

_Placeholder for complete performance comparison across all phases:_

| Phase | Training Time | Speedup (vs CPU) | Incremental Speedup | Memory Usage | Key Optimization |
|-------|---------------|------------------|---------------------|--------------|------------------|
| CPU Baseline | _TBD_ | 1.0× | - | _TBD_ | - |
| GPU Naive | _TBD_ | _TBD_ | _TBD_ | _TBD_ | Basic Parallelization |
| GPU Opt v1 | _TBD_ | _TBD_ | _TBD_ | _TBD_ | _TBD_ |
| GPU Opt v2 | _TBD_ | _TBD_ | _TBD_ | _TBD_ | _TBD_ |


## 3.2 Performance Visualizations


_Placeholder for visualizations:_
- Bar chart comparing training time across phases
- Line graph showing cumulative speedup progression
- Memory usage comparison
- Time breakdown by operation type

## 3.3 Performance Analysis Summary

_Placeholder for performance insights:_
- Which optimizations had the biggest impact?
- Where did we achieve the target speedup (> 20×)?
- What are the remaining bottlenecks?
- How does final performance compare to project objectives?

# **4. Lessons Learned and Challenges Overcome**

## 4.1 Key Technical Insights

_Placeholder for technical insights in these areas:_

**CUDA Programming:**
- Memory hierarchy and access patterns
- Thread synchronization and cooperation
- Occupancy and resource utilization
- Kernel launch configuration strategies

**Deep Learning Implementation:**
- Numerical stability considerations
- Gradient flow in deep networks
- Memory-efficient backpropagation
- Weight initialization importance

**Performance Optimization:**
- Profiling-driven optimization methodology
- Trade-offs between memory and compute
- When to apply specific optimization techniques
- Diminishing returns from optimization

## 4.2 Major Challenges and Solutions

_Placeholder for challenge-solution pairs in this format:_

**Challenge 1: [Title]**
- **Problem:** _Describe the challenge faced_
- **Solution:** _How you solved it_
- **Lesson:** _What you learned_

**Challenge 2: [Title]**
- **Problem:** _Describe the challenge faced_
- **Solution:** _How you solved it_
- **Lesson:** _What you learned_

**Challenge 3: [Title]**
- **Problem:** _Describe the challenge faced_
- **Solution:** _How you solved it_
- **Lesson:** _What you learned_

_Add more challenges as needed_

## 4.3 Skills and Knowledge Gained

_Placeholder for:_
- Technical skills acquired
- Tools and techniques mastered
- Understanding deepened
- Practical experience gained


# **5. Conclusion and Future Work**

## 5.1 Project Summary

_Placeholder for project summary:_
- Brief recap of objectives
- Overview of implementation approach
- Summary of phases completed

## 5.2 Final Metrics and Achievements

**Performance Achievements:**

_Placeholder for final metrics:_
- Final training time: _TBD_
- Total speedup achieved: _TBD_
- Classification accuracy: _TBD_
- Memory efficiency: _TBD_

**Objectives Met:**
- ✓ Autoencoder training time: < 10 minutes (_actual: TBD_)
- ✓ Feature extraction time: < 20 seconds (_actual: TBD_)
- ✓ Classification accuracy: 60-65% (_actual: TBD_)
- ✓ GPU speedup: > 20× (_actual: TBD_)

## 5.3 Project Limitations

_Placeholder for limitations:_
- Current constraints and bottlenecks
- Simplifications made in implementation
- Trade-offs accepted
- What couldn't be achieved and why

## 5.4 Future Improvements and Extensions

**Potential Optimizations:**
- _List additional optimization opportunities_
- _Techniques not yet implemented_
- _Advanced CUDA features to explore_

**Architecture Enhancements:**
- Deeper encoder networks
- Different activation functions
- Batch normalization
- Residual connections

**Alternative Approaches:**
- End-to-end supervised learning
- Different loss functions
- Data augmentation
- Transfer learning from pretrained models

**Scalability:**
- Multi-GPU training
- Larger datasets
- Real-time inference optimization

## 5.5 Final Remarks

_Placeholder for final thoughts:_
- Overall experience and learning
- Value of the project
- Applicability to real-world problems
- Future directions for exploration

---

**End of Report**
